# Project 1: Exploratory Data Analysis (EDA)


## DecodeLabs Internship



# Continuation from: Data Cleaning & Preparation



# Input file: Cleaned_Orders_Dataset.csv

# Step 1: Import Required Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# # Step 2: Load the Cleaned Dataset

In [4]:
df = pd.read_csv("Cleaned_Orders_Dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])

print(df.head())

     OrderID       Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000 2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001 2024-08-23     C75739    Phone         2     151.35   
2  ORD200002 2024-02-27     C81728   Tablet         5     550.68   
3  ORD200003 2023-10-15     C33540    Chair         1     273.19   
4  ORD200004 2025-05-08     C81840  Printer         4     626.01   

  ShippingAddress PaymentMethod OrderStatus TrackingNumber  ItemsInCart  \
0     928 Main St    Debit Card     Shipped    TRK37947903            7   
1     823 Main St        Online     Shipped    TRK91186779            3   
2     512 Main St   Credit Card   Cancelled    TRK42903982            8   
3     275 Main St    Debit Card    Returned    TRK62788070            5   
4     668 Main St        Online   Delivered    TRK29241424            8   

  CouponCode ReferralSource  TotalPrice  
0     SAVE10      Instagram     2853.10  
1     SAVE10       Referral      302.70  
2   FREESHIP  

# # Step 3: Basic Statistics (Mean, Median, Count, Std)


In [5]:
numeric_cols = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

basic_stats = df[numeric_cols].agg(["count", "mean", "median", "std", "min", "max"])
print("\nBasic Statistics:\n", basic_stats)

# Mode is calculated separately since agg() does not support it directly
mode_stats = df[numeric_cols].mode().iloc[0]
print("\nMode of numeric columns:\n", mode_stats)



Basic Statistics:
            Quantity    UnitPrice  ItemsInCart   TotalPrice
count   1200.000000  1200.000000  1200.000000  1200.000000
mean       2.945833   356.412750     5.485000  1053.968300
median     3.000000   364.210000     5.000000   823.615000
std        1.407557   197.177146     2.281983   819.856558
min        1.000000    11.390000     1.000000    11.390000
max        5.000000   699.930000    10.000000  3456.400000

Mode of numeric columns:
 Quantity         1.00
UnitPrice      127.18
ItemsInCart      5.00
TotalPrice     211.14
Name: 0, dtype: float64


# Step 4: Categorical Column Frequency (Distribution of Categories)


In [6]:
categorical_cols = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource", "CouponCode"]

for col in categorical_cols:
    print(f"\nValue counts for {col}:\n", df[col].value_counts())


Value counts for Product:
 Product
Printer    181
Tablet     179
Chair      178
Laptop     173
Desk       170
Monitor    163
Phone      156
Name: count, dtype: int64

Value counts for PaymentMethod:
 PaymentMethod
Online         258
Cash           246
Credit Card    234
Debit Card     232
Gift Card      230
Name: count, dtype: int64

Value counts for OrderStatus:
 OrderStatus
Cancelled    250
Returned     247
Pending      237
Shipped      235
Delivered    231
Name: count, dtype: int64

Value counts for ReferralSource:
 ReferralSource
Instagram    259
Email        250
Google       241
Facebook     228
Referral     222
Name: count, dtype: int64

Value counts for CouponCode:
 CouponCode
FREESHIP     313
No Coupon    309
WINTER15     292
SAVE10       286
Name: count, dtype: int64


# # Step 5: Trend Analysis Over Time


In [7]:
df["YearMonth"] = df["Date"].dt.to_period("M")
monthly_sales = df.groupby("YearMonth")["TotalPrice"].sum()

plt.figure(figsize=(10, 5))
monthly_sales.plot(kind="line", marker="o")
plt.title("Monthly Sales Trend (TotalPrice)")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.savefig("monthly_sales_trend.png")
plt.close()

# Step 6: Distribution Shape (Skewness and Kurtosis)


In [8]:
for col in numeric_cols:
    print(f"\n{col} -> Skewness: {df[col].skew():.3f}, Kurtosis: {df[col].kurt():.3f}")


Quantity -> Skewness: 0.028, Kurtosis: -1.295

UnitPrice -> Skewness: -0.027, Kurtosis: -1.191

ItemsInCart -> Skewness: 0.001, Kurtosis: -0.709

TotalPrice -> Skewness: 0.891, Kurtosis: -0.040


# Step 7: Histograms (Geometry of Distribution)


In [9]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.savefig("distributions.png")
plt.close()

# Step 8: Outlier Detection - IQR Method


In [10]:
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)], lower, upper

for col in numeric_cols:
    outliers, lower, upper = iqr_outliers(df[col])
    print(f"\n{col} - IQR bounds: [{lower:.2f}, {upper:.2f}] -> {len(outliers)} outliers found")


Quantity - IQR bounds: [-1.00, 7.00] -> 0 outliers found

UnitPrice - IQR bounds: [-317.20, 1024.83] -> 0 outliers found

ItemsInCart - IQR bounds: [-0.50, 11.50] -> 0 outliers found

TotalPrice - IQR bounds: [-1341.41, 3330.41] -> 8 outliers found


# Step 9: Outlier Detection - Z-Score Method


In [11]:
from scipy import stats

for col in numeric_cols:
    z_scores = np.abs(stats.zscore(df[col]))
    z_outliers = df[col][z_scores > 3]
    print(f"\n{col} - Z-score method -> {len(z_outliers)} outliers found (threshold = 3)")


Quantity - Z-score method -> 0 outliers found (threshold = 3)

UnitPrice - Z-score method -> 0 outliers found (threshold = 3)

ItemsInCart - Z-score method -> 0 outliers found (threshold = 3)

TotalPrice - Z-score method -> 0 outliers found (threshold = 3)


# Step 10: Boxplots (Visual Outlier Check)


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(x=df[col], ax=ax, color="skyblue")
    ax.set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.savefig("boxplots.png")
plt.close()

# Step 11: Correlation Analysis


In [13]:
correlation_matrix = df[numeric_cols].corr()
print("\nCorrelation Matrix:\n", correlation_matrix)

plt.figure(figsize=(6, 5))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")
plt.close()


Correlation Matrix:
              Quantity  UnitPrice  ItemsInCart  TotalPrice
Quantity     1.000000   0.014553     0.650061    0.615251
UnitPrice    0.014553   1.000000     0.000602    0.717081
ItemsInCart  0.650061   0.000602     1.000000    0.392540
TotalPrice   0.615251   0.717081     0.392540    1.000000


# Step 12: Group-wise Summary (Product vs Average TotalPrice)


In [14]:
product_summary = df.groupby("Product")["TotalPrice"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
print("\nProduct-wise Summary:\n", product_summary)


Product-wise Summary:
                 mean   median  count
Product                             
Laptop   1110.558150  915.640    173
Chair    1098.989382  928.575    178
Printer  1080.732652  857.400    181
Monitor  1077.616012  853.400    163
Tablet   1042.284637  786.660    179
Desk      985.058412  784.390    170
Phone     972.579423  691.715    156


# Step 13: OrderStatus vs Revenue


In [15]:
status_summary = df.groupby("OrderStatus")["TotalPrice"].sum().sort_values(ascending=False)
print("\nOrderStatus-wise Revenue:\n", status_summary)


OrderStatus-wise Revenue:
 OrderStatus
Cancelled    276396.21
Pending      256328.15
Shipped      246159.58
Returned     243277.70
Delivered    242600.32
Name: TotalPrice, dtype: float64


# Step 14: Key Observations Summary


In [16]:
print("\n--- Key Observations ---")
print(f"Total Orders: {len(df)}")
print(f"Average Order Value: {df['TotalPrice'].mean():.2f}")
print(f"Median Order Value: {df['TotalPrice'].median():.2f}")
print(f"Highest Selling Product (by avg revenue): {product_summary.index[0]}")
print(f"Most Common Payment Method: {df['PaymentMethod'].mode()[0]}")
print(f"Most Common Referral Source: {df['ReferralSource'].mode()[0]}")
print(f"Percentage of orders using a coupon: {(df['CouponCode'] != 'No Coupon').mean() * 100:.2f}%")

print("\nEDA completed. Charts saved as PNG files in the working directory.")


--- Key Observations ---
Total Orders: 1200
Average Order Value: 1053.97
Median Order Value: 823.62
Highest Selling Product (by avg revenue): Laptop
Most Common Payment Method: Online
Most Common Referral Source: Instagram
Percentage of orders using a coupon: 74.25%

EDA completed. Charts saved as PNG files in the working directory.
